In [ ]:
import pandas as pd
import hashlib
from sqlalchemy import create_engine

In [ ]:
import pandas as pd

df = pd.read_csv("25-26_Financial_year (1).csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

In [ ]:
print(df.head())

In [ ]:
 df.info()

In [ ]:
print(df.shape)

In [ ]:
print(df.describe(include="all"))

In [ ]:
print(df.isnull().sum())

In [ ]:
date_cols = [
    "BillingDate",
    "PaymentDate",
    "CreatedDate",
    "UpdatedDate"
]

for col in date_cols:
    df[col] = pd.to_datetime(
        df[col],
        errors="coerce"
    )

print(df[date_cols].dtypes)

In [ ]:
df["Party"] = df["Party"].fillna("Unknown")
df["Branch"] = df["Branch"].fillna("Unknown")
df["Department"] = df["Department"].fillna("Unknown")
df["AccountHead"] = df["AccountHead"].fillna("Unknown")

In [ ]:
df["Remarks"] = df["Remarks"].fillna("No Remarks")
df["Particulars"] = df["Particulars"].fillna("Not Mentioned")

In [ ]:
df["BillingDate"] = df["BillingDate"].fillna(
    df["PaymentDate"]
)

In [ ]:
print(
    "Duplicate rows before removal:",
    df.duplicated().sum()
)

In [ ]:
df.drop_duplicates(inplace=True)

print(
    "Duplicate rows after removal:",
    df.duplicated().sum()
)

In [ ]:
df.reset_index(drop=True, inplace=True)

print("Index reset successfully.")

In [ ]:
text_cols = df.select_dtypes(
    include="object"
).columns

for col in text_cols:
    df[col] = df[col].str.strip()

In [ ]:
df["AccountHead"] = df["AccountHead"].replace({
    "travel": "Travel",
    "service & Maintenance": "Service & Maintenance",
    "awards and rewards": "Awards and Rewards",
    "Medical Expens": "Medical Expense"
})

df["AccountHead"] = df["AccountHead"].str.strip()

In [ ]:
print(
    "Negative Amounts:",
    (df["Amount"] < 0).sum()
)

print(
    "Zero Amounts:",
    (df["Amount"] == 0).sum()
)

In [ ]:
print(
    "PaymentDate before BillingDate:",
    (df["PaymentDate"] < df["BillingDate"]).sum()
)

In [ ]:
df["Year"] = df["PaymentDate"].dt.year
df["Month"] = df["PaymentDate"].dt.month_name()
df["MonthNo"] = df["PaymentDate"].dt.month
df["Quarter"] = (
    "Q" + df["PaymentDate"].dt.quarter.astype(str)
)
df["DayName"] = df["PaymentDate"].dt.day_name()
df["Week"] = df["PaymentDate"].dt.isocalendar().week

In [ ]:
df["PaymentDelay"] = (
    df["PaymentDate"] - df["BillingDate"]
).dt.days

In [ ]:
df["PaymentStatus"] = df["PaymentDelay"].apply(
    lambda x:
        "Early" if x < 0
        else ("On Time" if x == 0 else "Delayed")
)

In [ ]:
df["MonthYear"] = (
    df["PaymentDate"]
    .dt.to_period("M")
    .astype(str)
)

In [ ]:
df["SignedAmount"] = df.apply(
    lambda row:
        -row["Amount"]
        if row["Type"] == "Debit"
        else row["Amount"],
    axis=1
)

In [ ]:
Q1 = df["Amount"].quantile(0.25)
Q3 = df["Amount"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df["Amount"] < lower_bound) |
    (df["Amount"] > upper_bound)
]

print("Potential outliers:", len(outliers))

In [ ]:
source_key_columns = [
    "Amount",
    "Type",
    "PaymentMode",
    "Party",
    "Branch",
    "Department",
    "Account",
    "AccountHead",
    "Remarks",
    "BillingDate",
    "PaymentDate",
    "CreatedDate",
    "UpdatedDate",
    "Adjustment",
    "SelfTransfer",
    "Particulars"
]

In [ ]:
def create_transaction_key(row):
    value = "|".join(
        str(row[col])
        for col in source_key_columns
    )

    return hashlib.sha256(
        value.encode("utf-8")
    ).hexdigest()

In [ ]:
df["TransactionKey"] = df.apply(
    create_transaction_key,
    axis=1
)

print("Total records:", len(df))
print(
    "Unique TransactionKeys:",
    df["TransactionKey"].nunique()
)

In [ ]:
duplicate_keys = df[
    df["TransactionKey"].duplicated(
        keep=False
    )
]

print(
    "Rows with duplicate TransactionKeys:",
    len(duplicate_keys)
)

In [ ]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///financial_analysis.db")

print("SQLite connection successful!")

In [ ]:
existing_keys_df = pd.read_sql(
    """
    SELECT TransactionKey
    FROM financial_transactions
    """,
    engine
)

existing_keys = set(
    existing_keys_df["TransactionKey"]
)

print(
    "Existing database records:",
    len(existing_keys)
)

In [ ]:
new_records = df[
    ~df["TransactionKey"].isin(existing_keys)
].copy()

print("Incoming records:", len(df))
print(
    "New records detected:",
    len(new_records)
)
print(
    "Existing records skipped:",
    len(df) - len(new_records)
)

In [ ]:
if len(new_records) > 0:

    new_records.to_sql(
        "financial_transactions",
        engine,
        if_exists="append",
        index=False
    )

    print(
        len(new_records),
        "new records inserted."
    )

else:

    print(
        "No new records to insert."
    )

In [ ]:
result = pd.read_sql(
    """
    SELECT COUNT(*) AS total_records,
           COUNT(DISTINCT TransactionKey)
           AS unique_transaction_keys
    FROM financial_transactions
    """,
    engine
)

print(result)

In [ ]:
final_total = result.iloc[0]["total_records"]

print("=" * 50)
print("FINANCIAL DATA AUTOMATION COMPLETED")
print("=" * 50)

print("Incoming records:", len(df))
print(
    "Existing records skipped:",
    len(df) - len(new_records)
)
print(
    "New records inserted:",
    len(new_records)
)
print(
    "Total records in database:",
    final_total
)

print("=" * 50)

In [ ]:
import shutil
import os

source = r"financial_analysis.db"
backup = r"financial_analysis_backup.db"

shutil.copy2(source, backup)

print("Backup created successfully!")
print("Backup location:", backup)
print("Backup exists:", os.path.exists(backup))

In [ ]:
import sqlite3

conn = sqlite3.connect(db_path)

count = pd.read_sql(
    "SELECT COUNT(*) AS total_records FROM financial_transactions",
    conn
)

print(count)

conn.close()

In [ ]:
import pandas as pd

df = pd.read_csv("25-26_Financial_year (1).csv")

print("Raw records:", len(df))
print("Raw columns:", len(df.columns))

In [ ]:
print(df.columns.tolist())

In [ ]:
date_cols = [
    "BillingDate",
    "PaymentDate",
    "CreatedDate",
    "UpdatedDate"
]

for col in date_cols:
    df[col] = pd.to_datetime(
        df[col],
        errors="coerce"
    )

print("Date columns converted successfully.")
print(df[date_cols].dtypes)

In [ ]:
df["Party"] = df["Party"].fillna("Unknown")
df["Branch"] = df["Branch"].fillna("Unknown")
df["Department"] = df["Department"].fillna("Unknown")
df["AccountHead"] = df["AccountHead"].fillna("Unknown")

print("Missing categorical values handled successfully.")

In [ ]:
df["Remarks"] = df["Remarks"].fillna("No Remarks")
df["Particulars"] = df["Particulars"].fillna("Not Mentioned")

print("Missing text values handled successfully.")

In [ ]:
df["BillingDate"] = df["BillingDate"].fillna(df["PaymentDate"])

print("Missing BillingDate values handled successfully.")

In [ ]:
before_duplicates = len(df)

df.drop_duplicates(inplace=True)

after_duplicates = len(df)

print("Duplicate records removed:", before_duplicates - after_duplicates)
print("Records remaining:", after_duplicates)

In [ ]:
df.reset_index(drop=True, inplace=True)

print("Index reset successfully.")
print("Current number of records:", len(df))

In [ ]:
text_cols = [
    "Type",
    "PaymentMode",
    "Party",
    "Branch",
    "Department",
    "Account",
    "AccountHead",
    "Remarks",
    "Particulars"
]

for col in text_cols:
    df[col] = df[col].str.strip()

print("Text columns cleaned successfully.")


In [ ]:
df["AccountHead"] = df["AccountHead"].replace({
    "travel": "Travel",
    "service & Maintenance": "Service & Maintenance",
    "awards and rewards": "Awards and Rewards",
    "Medical Expens": "Medical Expense"
})

print("AccountHead values standardized successfully.")

In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nNegative amounts:", (df["Amount"] < 0).sum())

In [ ]:
df["Year"] = df["PaymentDate"].dt.year

df["Month"] = df["PaymentDate"].dt.month_name()

df["MonthNo"] = df["PaymentDate"].dt.month

df["Quarter"] = "Q" + df["PaymentDate"].dt.quarter.astype(str)

df["DayName"] = df["PaymentDate"].dt.day_name()

df["Week"] = df["PaymentDate"].dt.isocalendar().week.astype("Int64")

print("Date analysis columns created successfully.")

In [ ]:
df["PaymentDelay"] = (
    df["PaymentDate"] - df["BillingDate"]
).dt.days

print("PaymentDelay column created successfully.")
print(df["PaymentDelay"].describe())

In [ ]:
df["PaymentStatus"] = df["PaymentDelay"].apply(
    lambda x: "Early" if x < 0
    else "On Time" if x == 0
    else "Delayed"
)

print("PaymentStatus column created successfully.")
print(df["PaymentStatus"].value_counts())

In [ ]:
df["MonthYear"] = df["PaymentDate"].dt.to_period("M").astype(str)

print("MonthYear column created successfully.")
print(df["MonthYear"].head())

In [ ]:
df["SignedAmount"] = df.apply(
    lambda row: -row["Amount"]
    if row["Type"].strip().lower() == "debit"
    else row["Amount"],
    axis=1
)

print("SignedAmount column created successfully.")
print(df[["Amount", "Type", "SignedAmount"]].head())

In [ ]:
print("Final shape:", df.shape)

print("\nFinal columns:")
print(df.columns.tolist())

print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
import hashlib

source_key_columns = [
    "Amount",
    "Type",
    "PaymentMode",
    "Party",
    "Branch",
    "Department",
    "Account",
    "AccountHead",
    "Remarks",
    "BillingDate",
    "PaymentDate",
    "CreatedDate",
    "UpdatedDate",
    "Adjustment",
    "SelfTransfer",
    "Particulars"
]

def create_transaction_key(row):
    values = []

    for col in source_key_columns:
        value = row[col]

        if pd.isna(value):
            value = ""

        values.append(str(value))

    key_string = "|".join(values)

    return hashlib.sha256(
        key_string.encode("utf-8")
    ).hexdigest()

df["TransactionKey"] = df.apply(create_transaction_key, axis=1)

print("TransactionKey created successfully.")
print("Total records:", len(df))
print("Unique TransactionKeys:", df["TransactionKey"].nunique())


In [ ]:
duplicate_keys = df["TransactionKey"].duplicated().sum()

print("Duplicate TransactionKeys:", duplicate_keys)

if duplicate_keys == 0:
    print("✅ All TransactionKeys are unique.")
else:
    print("⚠️ Duplicate TransactionKeys found.")

In [ ]:
from sqlalchemy import create_engine

db_path = r"financial_analysis.db"

engine = create_engine(f"sqlite:///{db_path}")

print("SQLite database connected successfully!")
print("Database:", db_path)

In [ ]:
existing_keys_df = pd.read_sql(
    "SELECT TransactionKey FROM financial_transactions",
    engine
)

existing_keys = set(existing_keys_df["TransactionKey"])

print("Existing records in database:", len(existing_keys))

In [ ]:
from sqlalchemy import inspect

inspector = inspect(engine)

columns = inspector.get_columns("financial_transactions")

print("Current database columns:")
for col in columns:
    print(col["name"])

In [ ]:
from sqlalchemy import text

with engine.begin() as conn:
    conn.execute(
        text("""
        ALTER TABLE financial_transactions
        ADD COLUMN TransactionKey TEXT
        """)
    )

print("TransactionKey column added successfully.")

In [ ]:
# Prepare the existing database records
db_df = pd.read_sql(
    "SELECT * FROM financial_transactions",
    engine
)

# Create TransactionKey using the same function
db_df["TransactionKey"] = db_df.apply(
    create_transaction_key,
    axis=1
)

print("Database records:", len(db_df))
print("Unique TransactionKeys:", db_df["TransactionKey"].nunique())

In [ ]:
from sqlalchemy import text

with engine.begin() as conn:
    for _, row in db_df.iterrows():
        conn.execute(
            text("""
                UPDATE financial_transactions
                SET TransactionKey = :transaction_key
                WHERE rowid = :rowid
            """),
            {
                "transaction_key": row["TransactionKey"],
                "rowid": int(row.name) + 1
            }
        )

print("TransactionKeys populated successfully.")

In [ ]:
check_keys = pd.read_sql(
    """
    SELECT 
        COUNT(*) AS total_records,
        COUNT(TransactionKey) AS records_with_key,
        COUNT(DISTINCT TransactionKey) AS unique_keys
    FROM financial_transactions
    """,
    engine
)

print(check_keys)

In [ ]:
existing_keys = set(
    pd.read_sql(
        "SELECT TransactionKey FROM financial_transactions",
        engine
    )["TransactionKey"]
)

df["IsExisting"] = df["TransactionKey"].isin(existing_keys)

existing_count = df["IsExisting"].sum()
new_count = (~df["IsExisting"]).sum()

print("Incoming records:", len(df))
print("Existing records:", existing_count)
print("New records:", new_count)

In [ ]:
new_records = df[~df["IsExisting"]].copy()

print("New records ready for insertion:", len(new_records))

In [ ]:
# Get the first TransactionKey from the cleaned incoming data
incoming_key = df["TransactionKey"].iloc[0]

# Get the first TransactionKey stored in SQLite
db_key = pd.read_sql(
    """
    SELECT TransactionKey
    FROM financial_transactions
    WHERE TransactionKey IS NOT NULL
    LIMIT 1
    """,
    engine
)["TransactionKey"].iloc[0]

print("Incoming key:")
print(incoming_key)

print("\nDatabase key:")
print(db_key)

print("\nKeys match:", incoming_key == db_key)

In [ ]:
# Compare the first incoming record with the first database record
incoming = df.iloc[0]

db_record = pd.read_sql(
    """
    SELECT *
    FROM financial_transactions
    LIMIT 1
    """,
    engine
).iloc[0]

for col in source_key_columns:
    print(f"\n--- {col} ---")
    print("Incoming :", repr(incoming[col]))
    print("Database :", repr(db_record[col]))

In [ ]:
def create_transaction_key(row):
    values = []

    for col in source_key_columns:
        value = row[col]

        if pd.isna(value):
            value = ""

        elif col in ["BillingDate", "PaymentDate", "CreatedDate", "UpdatedDate"]:
            value = pd.to_datetime(value).strftime("%Y-%m-%d")

        elif col in ["Adjustment", "SelfTransfer"]:
            value = int(bool(value))

        else:
            value = str(value).strip()


        values.append(str(value))

    key_string = "|".join(values)

    return hashlib.sha256(
        key_string.encode("utf-8")
    ).hexdigest()


print("TransactionKey function updated successfully.")

In [ ]:
df["TransactionKey"] = df.apply(
    create_transaction_key,
    axis=1
)

print("Incoming TransactionKeys regenerated.")
print("Total records:", len(df))
print("Unique TransactionKeys:", df["TransactionKey"].nunique())

In [ ]:
existing_keys = set(
    pd.read_sql(
        "SELECT TransactionKey FROM financial_transactions",
        engine
    )["TransactionKey"]
)

df["IsExisting"] = df["TransactionKey"].isin(existing_keys)

existing_count = df["IsExisting"].sum()
new_count = (~df["IsExisting"]).sum()

print("Incoming records:", len(df))
print("Existing records:", existing_count)
print("New records:", new_count)

In [ ]:
new_records = df[~df["IsExisting"]].copy()

print("New records ready for insertion:", len(new_records))

In [ ]:
# Create a test copy of 5 existing records
test_new_records = df.drop(columns=["IsExisting"]).head(5).copy()

# Make them genuinely different
test_new_records.loc[:, "Particulars"] = (
    test_new_records["Particulars"].astype(str) + " - AUTOMATION TEST"
)

# Regenerate TransactionKey after modification
test_new_records["TransactionKey"] = test_new_records.apply(
    create_transaction_key,
    axis=1
)

print("Test records created:", len(test_new_records))
print("Unique test keys:", test_new_records["TransactionKey"].nunique())

In [ ]:
test_existing_keys = existing_keys

test_new_records["IsExisting"] = (
    test_new_records["TransactionKey"].isin(test_existing_keys)
)

print("Test records:", len(test_new_records))
print("Detected as existing:", test_new_records["IsExisting"].sum())
print("Detected as new:", (~test_new_records["IsExisting"]).sum())

In [ ]:
test_new_records = test_new_records.drop(columns=["IsExisting"])

print("Test dataset prepared.")
print("Records:", len(test_new_records))
print("Columns:", len(test_new_records.columns))

In [ ]:
# Create a temporary test table
test_new_records.to_sql(
    "automation_test_transactions",
    engine,
    if_exists="replace",
    index=False
)

# Verify the test insertion
test_count = pd.read_sql(
    "SELECT COUNT(*) AS count FROM automation_test_transactions",
    engine
)

print("Test records inserted:", test_count["count"].iloc[0])

In [ ]:
from sqlalchemy import text

with engine.begin() as conn:
    conn.execute(
        text("DROP TABLE IF EXISTS automation_test_transactions")
    )

print("Temporary test table removed.")

In [ ]:
def process_financial_data(file_path, engine):

    # 1. Load new CSV
    data = pd.read_csv(file_path)

    # 2. Convert date columns
    date_cols = [
        "BillingDate",
        "PaymentDate",
        "CreatedDate",
        "UpdatedDate"
    ]

    for col in date_cols:
        data[col] = pd.to_datetime(
            data[col],
            errors="coerce"
        )

    # 3. Handle missing categorical values
    data["Party"] = data["Party"].fillna("Unknown")
    data["Branch"] = data["Branch"].fillna("Unknown")
    data["Department"] = data["Department"].fillna("Unknown")
    data["AccountHead"] = data["AccountHead"].fillna("Unknown")

    # 4. Handle missing text values
    data["Remarks"] = data["Remarks"].fillna("No Remarks")
    data["Particulars"] = data["Particulars"].fillna("Not Mentioned")

    # 5. Fill missing BillingDate
    data["BillingDate"] = data["BillingDate"].fillna(
        data["PaymentDate"]
    )

    # 6. Remove exact duplicates
    data.drop_duplicates(inplace=True)
    data.reset_index(drop=True, inplace=True)

    # 7. Remove unwanted spaces
    text_cols = [
        "Type",
        "PaymentMode",
        "Party",
        "Branch",
        "Department",
        "Account",
        "AccountHead",
        "Remarks",
        "Particulars"
    ]

    for col in text_cols:
        data[col] = data[col].str.strip()

    # 8. Standardize AccountHead
    data["AccountHead"] = data["AccountHead"].replace({
        "travel": "Travel",
        "service & Maintenance": "Service & Maintenance",
        "awards and rewards": "Awards and Rewards",
        "Medical Expens": "Medical Expense"
    })

    # 9. Create date analysis columns
    data["Year"] = data["PaymentDate"].dt.year
    data["Month"] = data["PaymentDate"].dt.month_name()
    data["MonthNo"] = data["PaymentDate"].dt.month
    data["Quarter"] = (
        "Q" + data["PaymentDate"].dt.quarter.astype(str)
    )
    data["DayName"] = data["PaymentDate"].dt.day_name()
    data["Week"] = data["PaymentDate"].dt.isocalendar().week.astype("Int64")

    # 10. Payment delay
    data["PaymentDelay"] = (
        data["PaymentDate"] - data["BillingDate"]
    ).dt.days

    # 11. Payment status
    data["PaymentStatus"] = data["PaymentDelay"].apply(
        lambda x: "Early" if x < 0
        else "On Time" if x == 0
        else "Delayed"
    )

    # 12. MonthYear
    data["MonthYear"] = (
        data["PaymentDate"]
        .dt.to_period("M")
        .astype(str)
    )

    # 13. SignedAmount
    data["SignedAmount"] = data.apply(
        lambda row: -row["Amount"]
        if row["Type"].strip().lower() == "debit"
        else row["Amount"],
        axis=1
    )

    # 14. Create TransactionKey
    data["TransactionKey"] = data.apply(
        create_transaction_key,
        axis=1
    )

    # 15. Read existing keys
    existing_keys = set(
        pd.read_sql(
            "SELECT TransactionKey FROM financial_transactions",
            engine
        )["TransactionKey"]
    )

    # 16. Identify new records
    data["IsExisting"] = data["TransactionKey"].isin(
        existing_keys
    )

    new_records = data[
        ~data["IsExisting"]
    ].copy()

    # 17. Remove helper column
    new_records.drop(
        columns=["IsExisting"],
        inplace=True
    )

    # 18. Insert only new records
    if len(new_records) > 0:
        new_records.to_sql(
            "financial_transactions",
            engine,
            if_exists="append",
            index=False
        )

    print("Incoming records:", len(data))
    print("Existing records skipped:", data["IsExisting"].sum())
    print("New records inserted:", len(new_records))

    return new_records

In [ ]:
raw_file_path = "25-26_Financial_year (1).csv"

new_records = process_financial_data(
    raw_file_path,
    engine
)

In [ ]:
final_check = pd.read_sql(
    """
    SELECT COUNT(*) AS total_records
    FROM financial_transactions
    """,
    engine
)

print(final_check)

In [ ]:
final_verification = pd.read_sql(
    """
    SELECT
        COUNT(*) AS total_records,
        COUNT(TransactionKey) AS records_with_key,
        COUNT(DISTINCT TransactionKey) AS unique_keys
    FROM financial_transactions
    """,
    engine
)

print(final_verification)

if (
    final_verification.loc[0, "total_records"] == 15826
    and final_verification.loc[0, "records_with_key"] == 15826
    and final_verification.loc[0, "unique_keys"] == 15826
):
    print("\n✅ FINAL VERIFICATION PASSED")
else:
    print("\n⚠️ CHECK REQUIRED")

In [ ]:
import shutil
import os

db_path = r"financial_analysis.db"
backup_path = r"financial_analysis_backup.db"

# Create a safety copy of the current database first
current_backup = r"financial_analysis_before_restore.db"
shutil.copy2(db_path, current_backup)

# Restore the known-good backup
shutil.copy2(backup_path, db_path)

print("Database restored successfully!")
print("Current database:", db_path)
print("Safety copy:", current_backup)

In [ ]:
import sqlite3
import pandas as pd

db_path = r"financial_analysis.db"

conn = sqlite3.connect(db_path)

result = pd.read_sql(
    "SELECT COUNT(*) AS total_records FROM financial_transactions",
    conn
)

print(result)

conn.close()

In [ ]:
import os

db_path = r"financial_analysis.db"
backup_path = r"financial_analysis_backup.db"

print("Main DB size:", os.path.getsize(db_path))
print("Backup DB size:", os.path.getsize(backup_path))

In [ ]:
import sqlite3
import pandas as pd

backup_path = r"financial_analysis_backup.db"

conn = sqlite3.connect(backup_path)

result = pd.read_sql(
    "SELECT COUNT(*) AS total_records FROM financial_transactions",
    conn
)

print(result)

conn.close()

In [ ]:
import sqlite3
import pandas as pd

db_path = r"financial_analysis.db"
csv_path = r"Financial_Data_Cleaned.csv"

# Read the correct cleaned data
df_clean = pd.read_csv(csv_path)

print("CSV records:", len(df_clean))

# Connect to SQLite
conn = sqlite3.connect(db_path)

# Replace the duplicated table with the correct 15,826 records
df_clean.to_sql(
    "financial_transactions",
    conn,
    if_exists="replace",
    index=False
)

# Verify
result = pd.read_sql(
    "SELECT COUNT(*) AS total_records FROM financial_transactions",
    conn
)

print(result)

conn.close()

In [ ]:
import sqlite3
import pandas as pd

db_path = r"financial_analysis.db"

conn = sqlite3.connect(db_path)

columns = pd.read_sql(
    "PRAGMA table_info(financial_transactions)",
    conn
)

print(columns[["name", "type"]].to_string(index=False))

conn.close()

In [ ]:
import sqlite3
import pandas as pd

db_path = r"financial_analysis.db"

conn = sqlite3.connect(db_path)

check = pd.read_sql(
    "SELECT COUNT(*) AS total_records FROM financial_transactions",
    conn
)

print(check)

conn.close()

In [ ]:
import shutil
import os

db_path = r"financial_analysis.db"

good_backup = r"financial_analysis_15826_backup.db"

shutil.copy2(db_path, good_backup)

print("Good backup created successfully!")
print("Backup:", good_backup)
print("Exists:", os.path.exists(good_backup))